# Temporal Video Grounding Assistant
**Cherisha Killari Giribabu · Panther ID: 002901729**

This prototype uses MobileVideoGPT to check short video clips against a text query and return candidate timestamps.

Run the cells in order in **Google Colab with a GPU runtime**. You will need `Video_Test_Clips.zip` for the action tests and `colab_interactive_demo.py` for the interactive demo. The demo accepts your own video.

Saved outputs have been cleared. This notebook keeps the fixes from the working session; it has not been rerun on a GPU during this cleanup. Predictions can contain false positives, and timestamps follow approximately three-second clip boundaries.


## 1. Check the runtime

In [ ]:
import os
import re
import subprocess
import sys
import time
import zipfile
from pathlib import Path
from google.colab import files


def run_command(command, **options):
    return subprocess.run(command, check=True, **options)


print("Notebook Python:", sys.version.split()[0])
run_command(["nvidia-smi"])


## 2. Create the model environment and install dependencies
The model runs in a separate Python 3.11 environment.

In [ ]:
# Create a Python 3.11 environment for the model.
run_command(
    [sys.executable, "-m", "pip", "install", "-q", "uv"],
)

run_command(
    [sys.executable, "-m", "uv", "venv",
     "--python", "3.11", "--seed", "/content/video_env"],
)

model_python = "/content/video_env/bin/python"
run_command([model_python, "--version"])
# Install the PyTorch versions used by the model.
run_command(
    [
        model_python, "-m", "pip", "install",
        "torch==2.1.2",
        "torchvision==0.16.2",
        "--index-url", "https://download.pytorch.org/whl/cu118",
    ],
)

run_command(
    [model_python, "-m", "pip", "install",
     "transformers==4.41.0", "numpy<2"],
)

# Confirm that this environment can access the GPU.
run_command(
    [
        model_python, "-c",
        "import torch; "
        "print('PyTorch:', torch.__version__); "
        "print('GPU available:', torch.cuda.is_available())",
    ],
)


## 3. Download the model code and install video packages

In [ ]:
repo_folder = Path("/content/Mobile-VideoGPT")

if not repo_folder.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/Amshaker/Mobile-VideoGPT",
            str(repo_folder),
        ],
        check=True,
    )

print("Model code downloaded.")
packages = [
    "numpy<2",
    "opencv-python-headless==4.10.0.84",
    "imageio",
    "decord==0.6.0",
    "mmengine==0.10.7",
    "einops",
    "timm==0.9.16",
    "accelerate==0.30.1",
    "sentencepiece",
    "ninja",
    "packaging",
    "wheel",
    "setuptools<75",
]

subprocess.run(
    [model_python, "-m", "pip", "install", *packages],
    check=True,
)

print("Video-processing packages installed.")


## 4. Set up VideoMamba and the CUDA driver
Prebuilt extensions match PyTorch 2.1 and CUDA 11.8. The driver path is detected from the library loaded by PyTorch.

In [ ]:
model_python = "/content/video_env/bin/python"
model_folder = Path("/content/Mobile-VideoGPT")
video_mamba_folder = Path("/content/VideoMamba")
test_folder = Path("/content/video_test")


# Install prebuilt packages matching our PyTorch environment.
wheel_urls = [
    "https://github.com/Dao-AILab/causal-conv1d/releases/download/"
    "v1.0.0/causal_conv1d-1.0.0%2Bcu118torch2.1cxx11abiFALSE"
    "-cp311-cp311-linux_x86_64.whl",

    "https://github.com/state-spaces/mamba/releases/download/"
    "v1.0.1/mamba_ssm-1.0.1%2Bcu118torch2.1cxx11abiFALSE"
    "-cp311-cp311-linux_x86_64.whl",
]

print("Installing VideoMamba's CUDA packages...")
for url in wheel_urls:
    run_command([
        model_python, "-m", "pip", "install", "--no-deps", url
    ])

# VideoMamba uses its own bidirectional Mamba implementation.
if not video_mamba_folder.exists():
    run_command([
        "git", "clone", "--depth", "1",
        "https://github.com/OpenGVLab/VideoMamba",
        str(video_mamba_folder),
    ])

environment = os.environ.copy()
environment["PYTHONPATH"] = (
    f"{video_mamba_folder / 'mamba'}:{model_folder}"
)


# Locate the driver library actually loaded by PyTorch.
driver_check = """
import torch
from pathlib import Path

torch.cuda.init()

for line in Path('/proc/self/maps').read_text().splitlines():
    if 'libcuda.so' in line:
        print(line.split()[-1])
        break
"""

result = subprocess.run(
    [model_python, "-c", driver_check],
    capture_output=True,
    text=True,
    check=True,
)

paths = [
    line.strip() for line in result.stdout.splitlines()
    if line.strip().startswith("/") and "libcuda.so" in line
]

if not paths:
    raise RuntimeError(
        "Could not locate the loaded driver:\n" + result.stderr
    )

driver = Path(paths[0]).resolve()
driver_link = driver.parent / "libcuda.so"

if not driver_link.exists():
    driver_link.symlink_to(driver)

# Make this directory visible to Triton's library lookup.
Path("/etc/ld.so.conf.d/video-model.conf").write_text(
    str(driver.parent) + "\n"
)
subprocess.run(["ldconfig"], check=True)

print("Driver located:", driver)

environment = os.environ.copy()
environment["PYTHONPATH"] = (
    "/content/VideoMamba/mamba:/content/Mobile-VideoGPT"
)
environment["LD_LIBRARY_PATH"] = (
    str(driver.parent) + ":" +
    environment.get("LD_LIBRARY_PATH", "")
)


print("Checking model imports...")
run_command(
    [
        model_python, "-c",
        "import torch; "
        "import causal_conv1d_cuda; "
        "import selective_scan_cuda; "
        "from mobilevideogpt.utils import preprocess_input; "
        "print('Model imports passed. GPU:', torch.cuda.is_available())",
    ],
    cwd=model_folder,
    env=environment,
)


## 5. Upload the test clips
Select `Video_Test_Clips.zip`. Colab may add a number to the uploaded filename; the code accepts that.

In [ ]:
print("Select Video_Test_Clips.zip below.")
uploaded = files.upload()

archive_names = [
    name for name in uploaded
    if name.lower().endswith(".zip")
]
if len(archive_names) != 1:
    raise ValueError("Upload only Video_Test_Clips.zip.")

test_folder.mkdir(exist_ok=True)

with zipfile.ZipFile(archive_names[0]) as archive:
    for member in archive.infolist():
        destination = (test_folder / member.filename).resolve()
        if not destination.is_relative_to(test_folder.resolve()):
            raise ValueError("Invalid ZIP path.")
    archive.extractall(test_folder)


required_paths = [test_folder / "run_model.py", test_folder / "prepared"]
if not all(path.exists() for path in required_paths):
    raise FileNotFoundError("The ZIP must contain run_model.py and the prepared folder.")
print("Test clips are ready.")


## 6. Run the three action tests
Two queries describe visible actions; the drinking query is an absent-action control. The results are predictions, not verified accuracy scores.

In [ ]:
tests = [
    ("The person removes the lid from the bottle",
     "bottle_lid_results.json"),
    ("The person opens the book",
     "book_results.json"),
    ("The person drinks from the bottle",
     "absent_action_results.json"),
]

for event, filename in tests:
    print("\nTesting:", event, flush=True)

    result = subprocess.run(
        [
            model_python,
            "/content/video_test/run_model.py",
            "--prepared", "/content/video_test/prepared",
            "--event", event,
            "--out", f"/content/video_test/{filename}",
        ],
        cwd="/content/Mobile-VideoGPT",
        env=environment,
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-6000:])
        raise RuntimeError(f"Model test failed: {event}. See the output above.")

    print(result.stdout[-2500:])

results_zip = Path("/content/Video_Model_Results.zip")

with zipfile.ZipFile(
    results_zip, "w", zipfile.ZIP_DEFLATED
) as archive:
    for _, filename in tests:
        archive.write(
            Path("/content/video_test") / filename,
            filename,
        )

print("All three tests finished.")
files.download(str(results_zip))


## 7. Install the demo packages and upload the demo script
The pinned web packages and `Agg` backend address the launch errors from the original session.

In [ ]:
run_command([
    model_python, "-m", "pip", "install",
    "gradio==4.44.1",
    "pydantic==2.10.6",
    "huggingface_hub==0.25.2",
    "fastapi==0.115.6",
    "starlette==0.41.3",
])

print("Select colab_interactive_demo.py.")
uploaded = files.upload()
demo_names = [
    name for name in uploaded
    if Path(name).stem.startswith("colab_interactive_demo")
    and Path(name).suffix.lower() == ".py"
]
if len(demo_names) != 1:
    raise ValueError("Upload one colab_interactive_demo Python script.")

demo_path = Path("/content/colab_interactive_demo.py")
demo_path.write_bytes(uploaded[demo_names[0]])
print("Demo script is ready.")


## 8. Launch the interactive demo
Open the printed link, upload your video, and ask **When is the water bottle visible?** Keep the Colab runtime running while using the demo.

In [ ]:
environment = os.environ.copy()
environment["MPLBACKEND"] = "Agg"
environment["PYTHONPATH"] = (
    "/content/VideoMamba/mamba:/content/Mobile-VideoGPT"
)
environment["LD_LIBRARY_PATH"] = (
    str(driver.parent) + ":" +
    environment.get("LD_LIBRARY_PATH", "")
)
environment["PYTHONUNBUFFERED"] = "1"

if "demo_process" in globals() and demo_process.poll() is None:
    demo_process.terminate()
    try:
        demo_process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        demo_process.kill()
        demo_process.wait()

if "demo_log" in globals():
    demo_log.close()

log_path = Path("/content/interactive_demo.log")
demo_log = log_path.open("w")

demo_process = subprocess.Popen(
    [
        "/content/video_env/bin/python",
        "/content/colab_interactive_demo.py",
    ],
    cwd="/content/Mobile-VideoGPT",
    env=environment,
    stdout=demo_log,
    stderr=subprocess.STDOUT,
)

print("Starting the demo...")

for attempt in range(60):
    time.sleep(2)
    log_text = log_path.read_text()
    links = re.findall(r"https://[\w.-]+\.gradio\.live", log_text)

    if links:
        print("Open your demo:", links[-1])
        break

    if demo_process.poll() is not None:
        print(log_text[-6000:])
        raise RuntimeError("The demo stopped. See the log above.")
else:
    print(log_path.read_text()[-6000:])
    print("The link is not ready yet. Check the log above or rerun this cell.")


## Interpreting the output
The gallery contains midpoint previews of selected clips. It does not show object bounding boxes or generate a compressed highlight video. Review the timestamps and raw model answers, and download the JSON results from the demo. An incorrect selection should be reported as a false positive.